#### Set environment variables in [.env](.env) for LLM API calling

### Manual baseline conditions (REBUILD.md §4.0 / §4.1)

Three conditions that are not themselves prompt-optimization *techniques* (no search loop), evaluated directly:

1. **Zero-shot CoT** — manual floor. Task description + "Let's think step by step."
2. **Expert few-shot CoT** — human ceiling. The BBH benchmark authors' own three worked chain-of-thought exemplars (`BIG-Bench-Hard/cot-prompts/<task>.txt`).
3. **Free rewrite** — rubric-vs-rewriting control. Takes the same PromptWizard-optimized prompt MPIR refines and asks the model to improve it once, unstructured (no rubric, no multi-round validation), isolating MPIR's structured refinement from generic LLM rewriting ability (HANDOFF-GPU.md §6.4).

`GluePromptOpt` is constructed with the heuristic (MPIR) config purely as a lightweight vehicle for `evaluate()`'s prediction logging and `data_processor`/`setup_config` plumbing — no optimizer search is ever run in this notebook; `BEST_PROMPT` is set directly for each condition.

### Import Dependencies

In [ ]:
import sys
import os
import pickle
sys.path.insert(0, "../")
import promptwizard
from promptwizard.glue.promptopt.instantiate import GluePromptOpt
from promptwizard.glue.common.llm.llm_mgr import LLMMgr

from dotenv import load_dotenv
load_dotenv(override=True)


### Prepare the BBH task-specific dataset processor

In [ ]:
from bbh_processor import BBH

bbh_processor = BBH()


### Load the dataset\nSet `dataset_to_run` to one of the §4.0 gate tasks: `hyperbaton`, `ruin_names`, `penguins_in_a_table`. Re-run this notebook once per task (and, for the full grid, once per seed).

In [ ]:
dataset_to_run = 'hyperbaton'
seed = 42

from data_prep import prepare_bbh_task_split

split_paths = prepare_bbh_task_split(dataset_to_run, bbh_processor, seed=seed)
test_file_name = split_paths.test_file_name


### Construct a `GluePromptOpt` vehicle for `evaluate()`\nNo optimizer search runs here -- the heuristic (MPIR) config is reused purely because it has no task-specific fields, so it works unmodified for any task.

In [ ]:
gp = GluePromptOpt("configs/heuristic/promptopt_config.yaml",
                   "configs/heuristic/setup_config.yaml",
                   split_paths.train_file_name,
                   bbh_processor,
                   seed=seed)


### Condition 1: Zero-shot CoT (manual floor)

In [ ]:
from cot_prompts import task_description as get_task_description

# Manuscript Appendix A.1 preserves the exact zero-shot prompt used for
# penguins_in_a_table ("You are given a task that require [description].
# Let's think step by step. For each question, wrap only the final letter
# ... between <ANS_START> and <ANS_END> tags") -- the task-framing sentence
# is adopted here, but the delimiter instruction stays the general
# answer_format used by every other condition in this rebuild (a fixed
# letter-set enumeration per task doesn't generalize across all 23 tasks'
# varying answer shapes the way a general "wrap the final answer" does).
task_desc = get_task_description(dataset_to_run)
zero_shot_instruction = f"You are given the following task: {task_desc}\nLet's think step by step."
zero_shot_prompt = gp.prompt_opt.prompt_pool.final_prompt.format(
    instruction=zero_shot_instruction, few_shot_examples="",
    answer_format=gp.prompt_opt_param.answer_format)

gp.BEST_PROMPT = zero_shot_prompt
accuracy = gp.evaluate(test_file_name, task_name=dataset_to_run,
                       condition_name="zero_shot_cot", seed=seed)
print(f"Zero-shot CoT accuracy: {accuracy}")


### Condition 2: Expert few-shot CoT (human ceiling)

In [ ]:
from cot_prompts import expert_few_shot_prefix

few_shot_instruction = expert_few_shot_prefix(dataset_to_run)
few_shot_prompt = gp.prompt_opt.prompt_pool.final_prompt.format(
    instruction=few_shot_instruction, few_shot_examples="",
    answer_format=gp.prompt_opt_param.answer_format)

gp.BEST_PROMPT = few_shot_prompt
accuracy = gp.evaluate(test_file_name, task_name=dataset_to_run,
                       condition_name="expert_few_shot_cot", seed=seed)
print(f"Expert few-shot CoT accuracy: {accuracy}")


### Condition 3: Free rewrite (rubric-vs-rewriting control)

Uses the manuscript's own Appendix D meta-prompt (preserved in `manuscript/content_blocks.py` even though no implementing code ever existed elsewhere in the repo -- REBUILD.md §2.0). `{variant_instruction}` has no surviving definition anywhere in the manuscript source; left empty here as the only defensible default.

Requires `promptwizard.ipynb` to have already produced `results/promptwizard_<task>_seed<seed>.pkl` for this same `dataset_to_run`/`seed` -- run that notebook first if the cell below raises `FileNotFoundError`.

In [ ]:
pkl_path = f"results/promptwizard_{dataset_to_run}_seed{seed}.pkl"
with open(pkl_path, "rb") as f:
    starting_prompt = pickle.load(f)

# Manuscript Appendix D's exact meta-prompt. Note it requests the rewritten
# prompt directly ("Output only the rewritten task-solving prompt"), not
# delimiter-wrapped -- so unlike every other generation-style prompt in this
# codebase, the response is used as-is rather than parsed via a <START>/<END> pattern.
FREE_REWRITE_PROMPT = """You are given a task-solving prompt generated for a Big-Bench Hard
task.

Rewrite the prompt to improve clarity, readability,
and instruction organization while preserving the
original task meaning.

Preserve the original task, answer format, and examples.
Do not change the meaning of the task.
Do not add unrelated content.
Do not use any prompt-evaluation rubric.
Do not mention rubric criteria, scoring, strengths,
weaknesses, or feedback.

Output only the rewritten task-solving prompt.

{variant_instruction}

Original prompt:
<START>
{prompt}
<END>"""

rewrite_response = LLMMgr.chat_completion(
    [{"role": "user", "content": FREE_REWRITE_PROMPT.format(prompt=starting_prompt, variant_instruction="")}])
rewritten_instruction = rewrite_response.strip() or starting_prompt

free_rewrite_prompt = gp.prompt_opt.prompt_pool.final_prompt.format(
    instruction=rewritten_instruction, few_shot_examples="",
    answer_format=gp.prompt_opt_param.answer_format)

gp.BEST_PROMPT = free_rewrite_prompt
accuracy = gp.evaluate(test_file_name, task_name=dataset_to_run,
                       condition_name="free_rewrite", seed=seed)
print(f"Free rewrite accuracy: {accuracy}")
